Notebook extract de données databricks 


**import fichier csv** 

DEPUIS UN VOLUME



In [0]:
path = "/Volumes/databricks_gaelle/default/donnees_api/ent_comm_detail.csv"
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("delimiter", ";")
    .csv(path))
display(df)
df.printSchema()
print("Nombre de lignes :", df.count())

In [0]:
df.count()

voir les colonnes

In [0]:
df.columns

voir le contenu avec display(df)


In [0]:
display(df)

nettoyer les noms des colonnes

In [0]:
# Nettoyage des noms de colonnes

import re

colonnes = []
for c in df.columns:
    c = c.lower()
    c = re.sub(r'[^a-z0-9]', '_', c)
    c = re.sub(r'_+', '_', c)
    c = c.strip('_')    
    colonnes.append(c)

df = df.toDF(*colonnes)
print(df.columns)

Mettre le noms de colonnes en minuscules

In [0]:
df = df.toDF(*[c.lower() for c in df.columns])4

verification des noms de colonnes 

In [0]:
print(df.columns)

Créer une table Delta

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.ent_comm_detail")

Vérifier que la table existe

In [0]:
%sql
SELECT *
FROM default.ent_comm_detail
LIMIT 10;

Vérifier l'historique Delta

In [0]:
%sql

DESCRIBE HISTORY default.ent_comm_detail;

Faire un Time Travel

In [0]:
%sql

SELECT *
FROM default.ent_comm_detail
VERSION AS OF 0;

Créer la table Bronze

In [0]:

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "default.bronze_ent_comm_detail"
    )

verif  la table Bronze

In [0]:
%sql
SELECT *
FROM default.bronze_ent_comm_detail
LIMIT 10;

Explorer les données

types de colonnes

In [0]:
df.printSchema()

résumé statistique

In [0]:
df.summary().show()

Compter les valeurs manquantes :

In [0]:
from pyspark.sql.functions import col, count, when


df.select(
    [
        count(
            when(col(c).isNull(), True)
        ).alias(c)
        for c in df.columns
    ]
).show()

Chercher les doublons :

In [0]:
print(
    "Nombre lignes :",
    df.count()
)


print(
    "Après suppression doublons :",
    df.dropDuplicates().count()
)

Nettoyer les données

In [0]:
from pyspark.sql.functions import col

"""
df = df.withColumn(
    "population",
    col("population").cast("integer")
)"""

Supprimer les doublons

In [0]:
#df = df.dropDuplicates()

Supprimer les lignes invalides

In [0]:
#df = df.dropna()

Afficher les colonnes

In [0]:
df.printSchema()

Supprimer plusieurs colonnes

In [0]:
df = df.drop(
    "unnamed_0",
    "code_region",
    "ancienne_colonne"
)

In [0]:
print(df.columns)

Vérifier le résultat

In [0]:
df.summary().show()

Sauvegarde dans Silver

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.silver_ent_comm_detail")

Preparer la structure du silver

In [0]:
colonnes_a_garder = [
    "nom_de_la_commune",
    "code_postal",
    "siren",
    "nom_complet",
    "nom_raison_sociale"
]


df_silver = df.select(
    colonnes_a_garder
)

verifier le df_silver

In [0]:
display(df_silver)

supprimer les autres colonnes

écrire la table Silver

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "default.silver_ent_comm_detail"
    )

Vérification historique Delta

In [0]:
%sql
DESCRIBE DETAIL default.silver_ent_comm_detail;